In [2]:
import getpass
import os
import openai
import time
import random
from dotenv import load_dotenv
# from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
# from langchain_google_genai import ChatGoogleGenerativeAI
from typing import Callable, Any
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from openai import OpenAI

from langchain_postgres import PGVector
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.tools import tool
from langsmith import uuid7
from pydantic import BaseModel
import json

import tiktoken
from langchain_community.callbacks import get_openai_callback
import sqlalchemy

LANGSMITH_KEY = os.getenv('LANGSMITH_API_KEY')
HUGGINGFACE_KEY = os.getenv('HUGGINGFACE_KEY')
OPENAI_KEY = os.getenv('OPEN_AI_KEY')
CONNECTION_STRING = os.getenv('CONNECTION_KEY')
CONNECTION_STRING02 = os.getenv("CONNECTION_KEY02")
# GEMINI_KEY = os.getnev('GEMINI_KEY')

if LANGSMITH_KEY is not None and HUGGINGFACE_KEY is not None:
    print('Successfully retrieved keys')

c:\Users\panlw\Desktop\ML-helper-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully retrieved keys


In [3]:
# llm = HuggingFaceEndpoint(
#     repo_id="microsoft/Phi-3-mini-4k-instruct",
#     temperature=0.7,
#     huggingfacehub_api_token= HUGGINGFACE_KEY,
# )

model = ChatOpenAI(
    model="gpt-3.5-turbo", openai_api_key=OPENAI_KEY, temperature=0)

# chat_model = ChatHuggingFace(llm = llm)
#chat_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

if model:
    print('model detected')

model detected


In [4]:
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_KEY)

embeddings_v2 = OpenAIEmbeddings(
    openai_api_key=OPENAI_KEY,
    model="text-embedding-3-small"
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index = False,  # track index in original document
)


In [5]:
# loader = PyPDFLoader(r'documents\AAAMLP.pdf')
# docs = loader.load()
# docs = text_splitter.split_documents(docs)

In [6]:
# for doc in docs:
#     print(doc)
#     break

In [7]:
# page_lengths = [len(doc.page_content) for doc in docs]
# print(page_lengths)

import matplotlib.pyplot as plt

# lengths = [len(doc.page_content) for doc in docs]
# print(f"Chunks: {len(docs)}")
# print(f"Avg: {sum(lengths)/len(lengths):.0f} chars | Min: {min(lengths)} | Max: {max(lengths)}")
# plt.hist(lengths, bins=30); plt.xlabel("chars"); plt.show()

In [8]:
import psycopg2

def collection_has_data(connection_str, collection_name):
    try:

        conn = psycopg2.connect(connection_str)
        cur = conn.cursor()
        cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
        cur.execute("""
            SELECT COUNT(*) FROM langchain_pg_embedding e
            JOIN langchain_pg_collection c ON e.collection_id = c.uuid
            WHERE c.name = %s
        """, (collection_name,))
        count = cur.fetchone()[0]
        conn.close()
        return count > 0
    except:
        return False

connection_str = os.getenv('CONNECTION_KEY')
collection_name = 'ML knowledge'

In [9]:
conn = psycopg2.connect(CONNECTION_STRING)
cur = conn.cursor()
cur.execute("SELECT * FROM pg_available_extensions WHERE name = 'vector';")
result = cur.fetchone()
print(f"Extension details: {result}")
conn.close()

Extension details: ('vector', '0.8.2', '0.8.2', 'vector data type and ivfflat and hnsw access methods')


In [10]:
if collection_has_data(connection_str, collection_name):
    print(f'collection already exist -connecting to existing data')
    db = PGVector(
    embeddings=embeddings,
    # documents=docs,
    collection_name= collection_name,
    connection =connection_str,
)
    
else:
    print(f'collection empty, embedding and inserting documents')
    db = PGVector.from_documents(
    embedding=embeddings,
    documents=docs,
    collection_name= collection_name,
    connection =CONNECTION_STRING,
)
    print(f"Successfully stored {len(docs)} chunks in PostgreSQL.")


collection already exist -connecting to existing data


In [11]:
#testing second db

if collection_has_data(connection_str, 'ml_docs_v2'):
    print(f'collection already exist -connecting to existing data')
    db_v2 = PGVector(
    embeddings=embeddings,
    # documents=docs,
    collection_name= 'ml_docs_v2',
    connection =connection_str,
)
    
else:
    print(f'collection empty, embedding and inserting documents')
    db_v2 = PGVector.from_documents(
    embedding=embeddings,
    documents=docs,
    collection_name= 'ml_docs_v2',
    connection =CONNECTION_STRING,
)
    print(f"Successfully stored {len(docs)} chunks in PostgreSQL.")

# db_v2 = PGVector(
#     embeddings=embeddings_v2,
#     collection_name="ml_docs_v2",  # separate collection
#     connection=CONNECTION_STRING,
# )

collection already exist -connecting to existing data


In [12]:
# ingest data into second db

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,  # chunk size (characters)
#     chunk_overlap=200,  # chunk overlap (characters)
#     add_start_index = False,  # track index in original document
# )
# loader = PyPDFLoader(r'documents\AAAMLP.pdf')
# docs = loader.load()
# docs = text_splitter.split_documents(docs)
# # for doc in docs:
# #     print(doc)
# #     break

#db_v2.add_documents(docs)  # same docs variable from your original ingestion

In [13]:
import time

test_questions = [
'what is cross-validation',
'what is overfitting',
'what is stratified k-fold',
'what is supervised learning',
'what is k-fold cross-validation'
]

times = []
for q in test_questions:
    embedded_q = embeddings.embed_query(q)
    start = time.time()
    raw_results = db.similarity_search_with_score_by_vector(embedded_q)
    elapsed = (time.time() - start) * 1000 #ms
    times.append(elapsed)
    print(f"{q[:40]:40} | {elapsed: .2f}ms")

print(f'Average retrieval time: {sum(times)/len(times):.2f}ms')


what is cross-validation                 |  101.35ms
what is overfitting                      |  61.83ms
what is stratified k-fold                |  58.80ms
what is supervised learning              |  61.81ms
what is k-fold cross-validation          |  50.56ms
Average retrieval time: 66.87ms


In [14]:
#HNSW
engine = sqlalchemy.create_engine(CONNECTION_STRING)
with engine.connect() as connec:
    result = connec.execute(sqlalchemy.text(
        "SELECT extversion FROM pg_extension WHERE extname = 'vector'"
    ))
    for row in result:
        print(f"pgvector version: {row[0]}")
    

pgvector version: 0.8.2


In [15]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text(
    """
    ALTER TABLE  langchain_pg_embedding
    ALTER COLUMN embedding TYPE vector(1536)
    """

    ))
    conn.commit()
    print("column dimension set to 1536")

column dimension set to 1536


In [16]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        """
        SELECT column_name, data_type, udt_name
        FROM information_schema.columns
        WHERE table_name = 'langchain_pg_embedding'
        """
    ))
    for row in result:
        print(row)

('collection_id', 'uuid', 'uuid')
('embedding', 'USER-DEFINED', 'vector')
('cmetadata', 'jsonb', 'jsonb')
('id', 'character varying', 'varchar')
('document', 'character varying', 'varchar')


In [17]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text(
        """
        CREATE INDEX IF NOT EXISTS idx_embedding_hnsw 
        ON langchain_pg_embedding 
        USING hnsw (embedding vector_cosine_ops)
        WITH (m = 16, ef_construction = 64);
        """
    ))
    conn.commit()
    print("HNSW index created")

HNSW index created


In [18]:
dummy_vector = json.dumps([0.0]*1536)

with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        f"""
        EXPLAIN ANALYZE
        SELECT id, embedding <=> '{dummy_vector}'::vector(1536) AS distance
        FROM langchain_pg_embedding
        ORDER BY distance
        LIMIT 4;
        """
    ))
    for row in result:
        print(row[0])

Limit  (cost=260.28..278.13 rows=4 width=45) (actual time=7.772..8.597 rows=4.00 loops=1)
  Buffers: shared hit=415 read=65
  ->  Index Scan using idx_embedding_hnsw on langchain_pg_embedding  (cost=260.28..8247.80 rows=1790 width=45) (actual time=7.770..8.594 rows=4.00 loops=1)
        Order By: (embedding <=> '[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [19]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        "SELECT COUNT(*) FROM langchain_pg_embedding"
    ))
    for row in result:
        print(f"Total rows: {row[0]}")

Total rows: 1810


In [20]:

with engine.connect() as conn:
    conn.execute(sqlalchemy.text('SET enable_seqscan = off;'))
    result = conn.execute(sqlalchemy.text(
        f"""EXPLAIN ANALYZE
            SELECT id, embedding <=> '{dummy_vector}' :: vector(1536) AS distance
            FROM  langchain_pg_embedding
            ORDER  BY distance
            LIMIT 4;
        """
    ))

    for row in result:
        print(row[0])

Limit  (cost=260.28..278.13 rows=4 width=45) (actual time=0.409..0.426 rows=4.00 loops=1)
  Buffers: shared hit=451
  ->  Index Scan using idx_embedding_hnsw on langchain_pg_embedding  (cost=260.28..8247.80 rows=1790 width=45) (actual time=0.408..0.424 rows=4.00 loops=1)
        Order By: (embedding <=> '[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [21]:
engine = sqlalchemy.create_engine(CONNECTION_STRING)
dummy_vector = json.dumps([0.0]*1536)
for ef in [10, 40, 100, 200]:
    with engine.connect() as conn:
        conn.execute(sqlalchemy.text(f"SET hnsw.ef_search = {ef}"))
        result = conn.execute(sqlalchemy.text(f"""
            EXPLAIN ANALYZE
            SELECT id, embedding <=> '{dummy_vector}'::vector(1536) AS distance
            FROM langchain_pg_embedding
            ORDER BY distance
            LIMIT 4;
        """))
        rows = result.fetchall()
        # find the execution time line
        for row in rows:
            if 'Execution Time' in row[0]:
                print(f"ef_search={ef}: {row[0].strip()}")

ef_search=10: Execution Time: 62.657 ms
ef_search=40: Execution Time: 0.978 ms
ef_search=100: Execution Time: 12.341 ms
ef_search=200: Execution Time: 4.801 ms


In [22]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        """
            SELECT COUNT(*) FROM  langchain_pg_embedding
"""
    ))
    for row in result:
        print(f'total rows: {row[0]}')

total rows: 1810


In [23]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        "SELECT indexname FROM pg_indexes WHERE tablename = 'langchain_pg_embedding'"
    ))
    for row in result:
        print(row)

('langchain_pg_embedding_pkey',)
('ix_cmetadata_gin',)
('idx_embedding_hnsw',)


In [24]:
# db = PGVector.from_documents(
#     embedding=embeddings,
#     documents=docs,
#     collection_name='document1',
#     connection =CONNECTION_STRING,
# )

# print(f"Successfully stored {len(docs)} chunks in PostgreSQL.")

In [25]:
# with engine.connect() as conn:
#     conn.execute(sqlalchemy.text("TRUNCATE TABLE langchain_pg_embedding"))
#     conn.commit()
#     print("Table cleared")

# # verify
# with engine.connect() as conn:
#     result = conn.execute(sqlalchemy.text("SELECT COUNT(*) FROM langchain_pg_embedding"))
#     for row in result:
#         print(f"Rows after truncate: {row[0]}")

In [26]:
retriever = db.as_retriever()

In [27]:
@tool(description = 'content_and artifact')
def retrieve_context(query: str):
    retrieved_doc = db.similarity_search(query, k = 3)
    serialized = "\n\n".join(f"Source: {doc.metadata} \n Content: {doc.page_content}" for doc in retrieved_doc)

    return serialized, retrieved_doc




MAX_CONTEXT_TOKENS = 2000
def build_context_with_budget_cross_enc(results, threshold=0.3, max_tokens=MAX_CONTEXT_TOKENS):
    result = []
    total_tokens = 0
    enc = tiktoken.get_encoding("cl100k_base")
    for doc, score in results:
        chunk_tokens = len(enc.encode(doc.page_content))
        # if score > threshold:
        #     continue 

        if total_tokens + chunk_tokens > max_tokens:
            break
        total_tokens += chunk_tokens
        result.append((doc, score))

    return result

In [28]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# from langchain.evaluation.qa import QAGenerateChain, QAEvalChain

# template = """You are a helpful assistant. You will be provided with a document and a question.
# Your task is to answer the question based on the content of the document. 
# here are the relevant section: {context}
# Question: {question}
# These are the past messages between you and the user, please use this history list to answer the
# question as well.
# History: {history}
# """

MAX_HISTORY_TOKENS = 3000
MAX_CONTEXT_TOKENS = 1000 # TESTING

prompt = ChatPromptTemplate.from_messages([
    ("system", """
     RULES:
    1. Answer ONLY using the context below. 
    2. If the context does not contain the answer, you MUST set is_in_document to false and set answer to exactly: "This topic is not covered in the provided material."
    3. NEVER use outside knowledge. NEVER mention anything not in the context.
    4. If unsure, set is_in_document to false.

Context:
{context}
Context: {context}

You must respond with the following fields:
- topic: the main ML topic the question is about
- is_in_document: true only if the answer is found in the context above
- source_chunks_used: list of direct quotes from the context you used
- answer: your answer based solely on the context provided
"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])
# use_tools = [retrieve_context]
# agent = create_agent(model, use_tools, system_prompt= prompt) #set up template and the tools





In [29]:
# implement thread-level memory (short term memory)
store = {}
def get_section_history(session_id) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [30]:
def token_calculation(log, embedding_tokens, callback):
    embedding_rate = 0.1 # 0.1 USD per million input tokens embedding
    input_rate = 0.5 # 0.5 usd per million prompt token
    output_rate= 1.5 #1.5 USD per million output token
    print(f'inital question has {embedding_tokens} tokens')

    embedding_price = (embedding_tokens / 1e6)* embedding_rate
    input_cost = (callback.prompt_tokens/ 1e6) * input_rate
    output_cost = (callback.completion_tokens/1e6)*output_rate
    round_total = embedding_price + input_cost + output_cost


    log['embedding_tokens'] += embedding_tokens
    log['embedding_cost']   += embedding_price
    log['input_tokens']     += callback.prompt_tokens
    log['input_cost']       += input_cost
    log['output_tokens']    += callback.completion_tokens
    log['output_cost']      += output_cost
    log['total_tokens']     += callback.total_tokens
    log['total_cost']       += round_total

    return log
    



In [31]:
def num_tokens_from_strings(string, encoding_name):
    try:
        encoding = tiktoken.get_encoding(encoding_name)
        num_tokens = len(encoding.encode(string))
        print('num_token used for question', num_tokens)
        return num_tokens
    except KeyError:
        print('wrong encoding model')

    


In [32]:
def history_token_estimation(history: ChatMessageHistory, model: str):
    enc = tiktoken.encoding_for_model(model)
    total = 0
    for message in history.messages:
        print(f'message:{message}')
        total += len(enc.encode(message.content))
    return total




In [33]:
def daily_usage_check(user_log):
    session_budget = 0.5 # session budget set to 0.5 USD
    if user_log['total_cost'] > session_budget:
        return False

    return True
        # raise Exception("Session budget exceeded stopping the chat")

In [34]:
def pre_flight_check(messages, embed_model):
    nums_token_estimate = num_tokens_from_strings(messages, embed_model)
    if nums_token_estimate > MAX_CONTEXT_TOKENS:
        print(f"Question too long: {nums_token_estimate} tokens (max {MAX_CONTEXT_TOKENS})")
        return False

    return True


In [35]:
#exponential delay with jitter
async def async_exp_delay_jitter(max_attempts: int, fn: Callable, *args, **kwargs):
    result = None
    delay = 1
    last_exception = None
    for attempt in range(0, max_attempts):
        try: 
            result = fn(*args, **kwargs)
            return result
        except Exception as e:
           last_exception = e
           if isinstance(e,openai.AuthenticationError) or isinstance(e,openai.BadRequestError):
                raise ValueError(f"Bad request or authentication issue")
           
           elif isinstance(e, openai.RateLimitError) or isinstance(e, openai.APIStatusError) and e.status_code in (500, 503):
                temp_time = delay * (2**attempt) + random.random()
                print(f"Attempt:{attempt} Error: {e} Waiting time: {temp_time}")
                time.sleep(temp_time)
        #    elif isinstance(e, openai.APIStatusError) and e.status_code in (500, 503):
        #        temp_time = delay * (2**attempt) + random.random()
        #        print(f"Attempt:{attempt} Error: {e} Waiting time: {temp_time}")
        #        time.sleep(temp_time)
           else:
               raise e
           
        
    if result is None:
        raise last_exception
        
    return result

In [36]:
# structured LLM output (pydantic)
class LLMoutput(BaseModel):
    topic: str
    is_in_document: bool
    source_chunks_used: list[str]
    answer: str

class RAGResponse(BaseModel):
    answer: str
    topic: str
    is_in_document: bool
    source_title: str
    source_pages: list[int]
    confidence: str  #high, medium, low

class ChunkReference(BaseModel):
    page: int
    score: float
    snippet: str # take the first 100 cahrs for human readibilty


structured_model = model.with_structured_output(LLMoutput)


c:\Users\panlw\Desktop\ML-helper-RAG\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2380: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [ ]:
# class MockCrossEncoder:
#     def predict(self, pairs):
#         import random
#         return [random.uniform(0, 1) for _ in pairs]

# cross_enc_mod = MockCrossEncoder()

In [48]:
from sentence_transformers import CrossEncoder
cross_enc_mod = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

def cross_reranking(cross_model, query, retrieved_chunks):
    raw_pairs = [[query, chunk[0].page_content] for chunk in retrieved_chunks]
    scores = cross_model.predict(raw_pairs)
    original_docs = [chunk[0] for chunk in retrieved_chunks]
    paired_result = list(zip(original_docs, scores))
    sorted_results = sorted(paired_result, key = lambda x: x[1], reverse = True)

    return sorted_results

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8735.19it/s]


In [49]:

section_id = str(uuid7())

chain = prompt | structured_model
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_section_history,
    input_messages_key='question',
    history_messages_key='history'
)

log = {
    'embedding_tokens': 0,
    'embedding_cost': 0,
    'input_tokens': 0,
    'input_cost': 0,
    'output_tokens': 0,
    'output_cost': 0,
    'total_tokens': 0,
    'total_cost': 0
}

while True:
    print("--------------------------------------------")
    question = input("Enter your question (or 'exit' to quit): ")
    print(f'Human question:{question}')

    if question.lower() == 'exit':
        break

    if pre_flight_check(question, 'cl100k_base') is False:
        raise Exception('failed the pre-flight check')

    embedded_question = embeddings.embed_query(question)
    num_embed_tok = num_tokens_from_strings(question, "cl100k_base")

    raw_results = db.similarity_search_with_score_by_vector(embedded_question)

    # print raw scores so you can calibrate threshold

    # build chunk references 
    for doc, score in raw_results:
        print(f"Score: {score:.4f} | Page: {doc.metadata.get('page')}")
        print(f"Content: {doc.page_content[:300]}")
        print("---")
    
    chunk_reference = [
    {
        'page': doc.metadata.get("page"),
        'similarity score': round(score, 4),
        'snippet': doc.page_content[:100]
    }
    for doc, score in raw_results if score <= 0.3
]
        

    context = build_context_with_budget_cross_enc(raw_results, threshold=0.25, max_tokens=MAX_CONTEXT_TOKENS)
    # print(f'context_yes: {context}')
    sorted_context = cross_reranking(cross_enc_mod, question, context)
    extracted_context = [doc for doc, score in sorted_context]
    for doc, score in sorted_context:
        print(f"Reranker score: {score:.4f} | Page: {doc.metadata.get('page')}")


    if not chunk_reference:
        print("No relevant chunks found above threshold")

    for doc in chunk_reference:
        print(doc)

    current_history = get_section_history(section_id)
    history_tokens = history_token_estimation(current_history, model.model_name)

    if not daily_usage_check(log):
        print('You have reached the maximum tokens, output is truncated')
        print(f'history_tokens:{history_tokens}')
        break

    print(f'history_tokens:{history_tokens}')

    with get_openai_callback() as cb:
        result = await async_exp_delay_jitter(3, chain_with_history.invoke,
            {"question": question, "context": context},
            config={"configurable": {"session_id": section_id}}
        )
        print(f'RESULT testing: {dir(result)}') # testing, should return content ans result

        print(f"Input tokens:  {cb.prompt_tokens}")
        print(f"Output tokens: {cb.completion_tokens}")
        print(f"Total tokens:  {cb.total_tokens}")
        print(f"Cost:          ${cb.total_cost:.5f}")

        print(f'Answer: {result.answer}')
        print(f'Topic: {result.topic}')
        print(f'In document: {result.is_in_document}')
        print(f'Sources: {chunk_reference}')

        print('------')
        token_calculation(log, num_embed_tok, cb)

        print('log = ', log)
        log['last_response'] = {
            'answer': result.answer,
            'topic': result.topic,
            'is_in_document': result.is_in_document,
            'chunk_reference': chunk_reference
        }

    # write log to file
    with open('rag_logs.jsonl', 'a') as f:
        f.write(json.dumps({
            'session_id': section_id,
            'question': question,
            'answer': result.answer,
            'topic': result.topic,
            'is_in_document': result.is_in_document,
            'sources': chunk_reference,
            'tokens': {
                'input': cb.prompt_tokens,
                'output': cb.completion_tokens,
                'total': cb.total_tokens
            },
            'cost': cb.total_cost
        }) + '\n')

--------------------------------------------
Human question:exit


In [42]:
GOLDEN_DATASET = [
    {
        'question': 'What is cross-validation?',
        'expected_page': 21,  # page where answer lives
        'expected_topic': 'cross-validation'
    },
    {
        'question': 'What is overfitting?',
        'expected_page': 20,
        'expected_topic': 'overfitting'
    },
    {
        'question': 'What is k-fold cross-validation?',
        'expected_page': 21,
        'expected_topic': 'cross-validation'
    },
    {
        'question': 'What is supervised learning',
        'expected_page': 7,  # find this yourself from the book
        'expected_topic': 'supervised learning'
    },
    {
        'question': 'What is stratified k-fold?',
        'expected_page': 22,  # find this yourself
        'expected_topic': 'cross-validation'
    }
]

In [43]:
def evaluate_retrieval(golden_dataset, db, embeddings, thresholds =  0.3):
    results = []
    for item in golden_dataset:
        embedded_q = embeddings.embed_query(item['question'])
        raw_results = db.similarity_search_with_score_by_vector(embedded_q)
        retrieved_pages = [
            doc.metadata.get('page') 
            for doc, score in raw_results 
            if score <= thresholds
        ]
        
        hit = item['expected_page'] in retrieved_pages if item['expected_page'] else None
        
        results.append({
            'question': item['question'],
            'expected_page': item['expected_page'],
            'retrieved_pages': retrieved_pages,
            'hit': hit,
            'top_score': round(raw_results[0][1], 4) if raw_results else None
        })
    
    hits = [r for r in results if r['hit'] is True]
    total_scoreable = [r for r in results if r['hit'] is not None]
    recall = len(hits) / len(total_scoreable) if total_scoreable else 0
    
    print(f"\nRetrieval Recall@K: {recall:.0%} ({len(hits)}/{len(total_scoreable)})")
    print(f"Threshold: {thresholds}\n")
    
    for r in results:
        status = '✅' if r['hit'] else '❌' if r['hit'] is False else '⚠️'
        print(f"{status} Q: {r['question'][:50]}")
        print(f"   Expected page: {r['expected_page']} | Retrieved: {r['retrieved_pages']} | Top score: {r['top_score']}")
    
    return results

In [ ]:

def evalute_with_reranking(golden_dataset, embedding_model, db, cross_model, top_k, threshold):
    vanilla_hits = 0
    cross_hits = 0

    for item in golden_dataset:
        embedded_q = embedding_model.embed_query(item['question'])
        raw_results = db.similarity_search_with_score_by_vector(embedded_q, top_k)
        filtered = [(doc,score) for doc,score in raw_results if score <= threshold]
        sorted_results = cross_reranking(cross_model, item['question'], filtered)

        vanilla_pages = [doc.metadata.get('page') for doc,score in raw_results if score <= threshold]
        reranked_pages = [doc.metadata.get('page') for doc, score in sorted_results[:top_k]]

        if item['expected_page'] in vanilla_pages:
            vanilla_hits+=1
        if item['expected_page'] in reranked_pages:
            cross_hits += 1
        
    return f"Vanilla's recall@k: {vanilla_hits} | Cross encoder's recall@k: {cross_hits}"

In [45]:
result = evaluate_retrieval(GOLDEN_DATASET, db, embeddings, thresholds = 0.3)
print(result)


Retrieval Recall@K: 100% (5/5)
Threshold: 0.3

✅ Q: What is cross-validation?
   Expected page: 21 | Retrieved: [13, 21, 20, 28] | Top score: 0.1112
✅ Q: What is overfitting?
   Expected page: 20 | Retrieved: [19, 19, 20, 102] | Top score: 0.1287
✅ Q: What is k-fold cross-validation?
   Expected page: 21 | Retrieved: [21, 22, 25, 20] | Top score: 0.1243
✅ Q: What is supervised learning
   Expected page: 7 | Retrieved: [6, 7, 6, 8] | Top score: 0.152
✅ Q: What is stratified k-fold?
   Expected page: 22 | Retrieved: [22, 24, 25, 26] | Top score: 0.1311
[{'question': 'What is cross-validation?', 'expected_page': 21, 'retrieved_pages': [13, 21, 20, 28], 'hit': True, 'top_score': 0.1112}, {'question': 'What is overfitting?', 'expected_page': 20, 'retrieved_pages': [19, 19, 20, 102], 'hit': True, 'top_score': 0.1287}, {'question': 'What is k-fold cross-validation?', 'expected_page': 21, 'retrieved_pages': [21, 22, 25, 20], 'hit': True, 'top_score': 0.1243}, {'question': 'What is supervised 

In [ ]:
result = evaluate_retrieval(GOLDEN_DATASET, db_v2, embeddings_v2, thresholds = 0.45)
print(result)


Retrieval Recall@K: 60% (3/5)
Threshold: 0.45

❌ Q: What is cross-validation?
   Expected page: 21 | Retrieved: [20, 13, 28, 25] | Top score: 0.3495
✅ Q: What is overfitting?
   Expected page: 20 | Retrieved: [19, 19, 20] | Top score: 0.3703
✅ Q: What is k-fold cross-validation?
   Expected page: 21 | Retrieved: [22, 25, 22, 21] | Top score: 0.3099
❌ Q: What is supervised learning
   Expected page: 7 | Retrieved: [6] | Top score: 0.3739
✅ Q: What is stratified k-fold?
   Expected page: 22 | Retrieved: [23, 22, 22, 26] | Top score: 0.3571
[{'question': 'What is cross-validation?', 'expected_page': 21, 'retrieved_pages': [20, 13, 28, 25], 'hit': False, 'top_score': 0.3495}, {'question': 'What is overfitting?', 'expected_page': 20, 'retrieved_pages': [19, 19, 20], 'hit': True, 'top_score': 0.3703}, {'question': 'What is k-fold cross-validation?', 'expected_page': 21, 'retrieved_pages': [22, 25, 22, 21], 'hit': True, 'top_score': 0.3099}, {'question': 'What is supervised learning', 'expec

In [51]:
start = time.time()
eval_result = evalute_with_reranking(GOLDEN_DATASET, embeddings, db, cross_enc_mod, 4, 0.25)
print(eval_result)

mock_time = time.time() - start
print(f'time taken : {mock_time}')

Vanilla's recall@k: 5 | Cross encoder's recall@k: 5
time taken : 1.2602746486663818


In [ ]:
question = "what is cross-validation"
embedded_q = embeddings_v2.embed_query(question)
results = db_v2.similarity_search_with_score_by_vector(embedded_q)

for doc, score in results:
    print(f"Score: {score:.4f} | Page: {doc.metadata.get('page')}")
    print(f"Content: {doc.page_content[:150]}")
    print("---")

Score: 0.3794 | Page: 13
Content: Cross-validation  We did not build any models in the previous chapter. The reason for that is simple. Before creating any kind of machine learning mod
---
Score: 0.3824 | Page: 20
Content: Figure 3: Most general definition of overfitting.  Now we can go back to cross-validation.  While explaining about overfitting, I decided to divide th
---
Score: 0.3833 | Page: 28
Content: Cross-validation is the first and most essential step when it comes to building machine learning models. If you want to do feature engineering, split 
---
Score: 0.3921 | Page: 25
Content: same as the number of samples that we have in the dataset.   One should note that this type of cross-validation can be costly in terms of the time it 
---


In [ ]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(
        """
        SELECT e.cmetadata, e.document
        FROM langchain_pg_embedding e
        JOIN langchain_pg_collection c ON e.collection_id = c.uuid
        WHERE c.name = 'ml_docs_v2'
        AND e.cmetadata->>'page' = '20'
        """
    ))
    for row in result:
        print(row)

({'page': 20, 'title': 'Approaching (Almost) Any Machine Learning Problem', 'author': 'Abhishek Thakur', 'source': 'documents\\AAAMLP.pdf', 'creator':  ... (44 characters truncated) ... , 'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'page_label': '21', 'total_pages': 300, 'creationdate': "D:20210608080427Z00'00'"}, 'or may not apply to other datasets. However, there are a few types of cross-validation techniques which are the most popular and widely used.   These include:  • k-fold cross-validation • stratified k-fold cross-validation')
({'page': 20, 'title': 'Approaching (Almost) Any Machine Learning Problem', 'author': 'Abhishek Thakur', 'source': 'documents\\AAAMLP.pdf', 'creator':  ... (44 characters truncated) ... , 'producer': 'macOS Version 11.4 (Build 20F71) Quartz PDFContext', 'page_label': '21', 'total_pages': 300, 'creationdate': "D:20210608080427Z00'00'"}, 'Approaching (Almost) Any Machine Learning Problem – Abhishek Thakur \n 21 \nOccam’s razor in simp

In [ ]:
# Print the full retrieved chunks to see if it actually found the explanation
for i, (doc, score) in enumerate(results):
    print(f"Rank {i+1} | Score: {score:.4f} | Page: {doc.metadata.get('page')}")
    print(doc.page_content)
    print("="*50)

Rank 1 | Score: 0.3794 | Page: 13
Cross-validation  We did not build any models in the previous chapter. The reason for that is simple. Before creating any kind of machine learning model, we must know what cross-validation is and how to choose the best cross-validation depending on your datasets.  So, what is cross-validation, and why should we care about it?  We can find multiple definitions as to what cross-validation is. Mine is a one-liner: cross-validation is a step in the process of building a machine learning model which helps us ensure that our models fit the data accurately and also ensures that we do not overfit. But this leads to another term: overfitting.   To explain overfitting, I think it’s best if we look at a dataset. There is a red wine-quality dataset2 which is quite famous. This dataset has 11 different attributes that decide the quality of red wine.   These attributes include: • fixed acidity • volatile acidity • citric acid • residual sugar • chlorides • free sulf

In [ ]:
with engine.connect() as conn:
    # check row counts per collection
    result = conn.execute(sqlalchemy.text(
        """
        SELECT c.name, COUNT(e.id) as chunk_count
        FROM langchain_pg_collection c
        LEFT JOIN langchain_pg_embedding e ON e.collection_id = c.uuid
        GROUP BY c.name
        """
    ))
    for row in result:
        print(f"Collection: {row[0]} | Chunks: {row[1]}")

Collection: document1 | Chunks: 0
Collection: ml_docs_v2 | Chunks: 895
Collection: ML knowledge | Chunks: 895
Collection: AAAMLP.pdf | Chunks: 0
Collection: async_test_collection | Chunks: 15


In [ ]:
import asyncio
import nest_asyncio

async def embed_single(embeddings_model, text):
    lst = await embeddings_model.aembed_query(text)
    return lst

async def embed_all_async(embeddings_model, texts):
    # results = []
    coroutines = [embed_single(embeddings_model, text)for text in texts]
    results = await asyncio.gather(*coroutines)
    # results.append(result)
    return results

# async def embed_all_async(embeddings_model, texts):
#     results = []
#     result = await embeddings_model.aembed_documents(texts)
#     results.append(result)
#     return result

import time
texts = [
    "what is cross-validation",
    "what is overfitting",
    "what is neural networks",
    "what is gradient descent",
    "what is regularization"
]


start = time.time()
sync_embedding = [embeddings.embed_query(text)for text in texts]
sync_time = time.time() - start

print(f'Sync time: {sync_time:.3f}s')


start = time.time()
async_embeddings = await embed_all_async(embeddings, texts)
async_time = time.time() - start
print(f"Async time: {async_time:.3f}s")
print(f"Speedup: {sync_time/async_time:.1f}x")

Sync time: 0.714s
Async time: 0.210s
Speedup: 3.4x


In [ ]:
from sqlalchemy.ext.asyncio import create_async_engine
load_dotenv(override=True)
connection_str02 = os.getenv('CONNECTION_KEY_ASYNC')
aysnc_engine = create_async_engine(connection_str02)


if connection_str02:
    if "postgresql+asyncpg" not in connection_str02:
        # Replaces postgresql:// with postgresql+asyncpg://
        async_conn_str = connection_str02.replace("postgresql://", "postgresql+asyncpg://")
    else:
        async_conn_str = connection_str02

    # 3. Create the async engine
    async_engine = create_async_engine(async_conn_str)
    print("Async engine successfully created!")
else:
    print("Error: Base connection string not found.")

Async engine successfully created!


In [ ]:
db_async = PGVector(
    embeddings=embeddings,
    collection_name='async_test_collection',
    connection=async_engine  # test
)
print("Vector store re-initialized with the async engine.")

Vector store re-initialized with the async engine.


In [ ]:
async def push_and_query_db_query(vector_store, texts, embeddings):
    print("Starting async database insertion...")
    await vector_store.aadd_texts(texts = texts)
    print(f'successfully inserted into pgvector async')

    query = 'what is overfitting'
    print(f'seraching {query} async...')
    query_vector = await async_exp_delay_jitter(3, embed_single,embeddings, query)
    # query_vector = await embeddings.aembed_query(query)
    results = await vector_store.asimilarity_search_by_vector(query_vector, k = 2)
    return results


search_results = await push_and_query_db_query(db_async, texts, embeddings)

for i, doc in enumerate(search_results):
    print(f"Match {i+1}: {doc.page_content[:80]}")


Starting async database insertion...
successfully inserted into pgvector async
seraching what is overfitting async...
Match 1: what is overfitting
Match 2: what is overfitting


In [ ]:
async def benchmark_search(top_k, label):
    times = []
    total_results = 0

    
    for question in test_questions:
        embedded = await embed_single(embeddings, question)
        
        start = time.time()
        results = await db_async.asimilarity_search_with_score_by_vector(
            embedded, k=top_k
        )
        elapsed = time.time() - start
        
        passing = [r for r in results if r[1] <= 0.25]
        times.append(elapsed)
        total_results += len(passing)
    
    avg_time = sum(times) / len(times)
    avg_passing = total_results / len(test_questions)
    print(f"{label}: avg_time={avg_time*1000:.1f}ms | avg_passing_chunks={avg_passing:.1f}")


await benchmark_search(2,  "top_k=2 ")
await benchmark_search(4,  "top_k=4 ")
await benchmark_search(10, "top_k=10")
await benchmark_search(20, "top_k=20")

top_k=2 : avg_time=51.0ms | avg_passing_chunks=2.0
top_k=4 : avg_time=48.8ms | avg_passing_chunks=4.0
top_k=10: avg_time=50.7ms | avg_passing_chunks=7.6
top_k=20: avg_time=53.6ms | avg_passing_chunks=10.4


In [ ]:
# with engine.connect() as conn:
#     result = conn.execute(sqlalchemy.text("""

# select e.document, e.cmetadata
#  from langchain_pg_embedding e
# join langchain_pg_collection c on e.collection_id = c.uuid
# where c.name = 'ML knowledge'
# AND (e.cmetadata = '{}'::jsonb 
#              OR e.cmetadata IS NULL
#              OR NOT (e.cmetadata ? 'page'))                                  

# """))
#     rows = result.fetchall()
#     print(f'chunks without page metadata:{len(rows)}')
#     for row in rows:
#         print(row)

In [ ]:
# with engine.connect() as conn:
#     conn.execute(sqlalchemy.text("""
#         DELETE FROM langchain_pg_embedding e
#         USING langchain_pg_collection c
#         WHERE e.collection_id = c.uuid
#         AND c.name = 'ML knowledge'
#         AND (e.cmetadata = '{}'::jsonb 
#              OR e.cmetadata IS NULL
#              OR NOT (e.cmetadata ? 'page'))
#     """))
#     conn.commit()
#     print("Deleted chunks without page metadata")

In [ ]:
# build API endpoint
import subprocess
subprocess.run(["pip", "install", "fastapi", "uvicorn", "httpx"], 
               capture_output=True)

CompletedProcess(args=['pip', 'install', 'fastapi', 'uvicorn', 'httpx'], returncode=0, stdout=b'Requirement already satisfied: fastapi in c:\\users\\panlw\\appdata\\local\\programs\\python\\python310\\lib\\site-packages (0.136.1)\r\nRequirement already satisfied: uvicorn in c:\\users\\panlw\\appdata\\local\\programs\\python\\python310\\lib\\site-packages (0.46.0)\r\nRequirement already satisfied: httpx in c:\\users\\panlw\\appdata\\local\\programs\\python\\python310\\lib\\site-packages (0.28.1)\r\nRequirement already satisfied: starlette>=0.46.0 in c:\\users\\panlw\\appdata\\local\\programs\\python\\python310\\lib\\site-packages (from fastapi) (0.52.1)\r\nRequirement already satisfied: pydantic>=2.9.0 in c:\\users\\panlw\\appdata\\local\\programs\\python\\python310\\lib\\site-packages (from fastapi) (2.10.3)\r\nRequirement already satisfied: typing-extensions>=4.8.0 in c:\\users\\panlw\\appdata\\local\\programs\\python\\python310\\lib\\site-packages (from fastapi) (4.15.0)\r\nRequireme

In [ ]:
import requests

response = requests.post(
    'http://localhost:8000/query',
    json = {'question': 'what is cross validation'}
)
print(response.status_code)
print(response.json())

200
{'answer': 'Cross-validation is a step in the process of building a machine learning model that involves dividing training data into parts, training the model on some of these parts, and testing on the remaining parts. It helps ensure that models fit the data accurately and prevent overfitting.', 'topic': 'cross-validation', 'is_in_document': True, 'sources': [{'page': 13, 'score': 0.1583, 'snippet': 'Cross-validation  We did not build any models in the previous chapter. The reason for that is simple'}, {'page': 21, 'score': 0.1852, 'snippet': 'Approaching (Almost) Any Machine Learning Problem – Abhishek Thakur \n 22 \n• hold-out based validatio'}, {'page': 20, 'score': 0.188, 'snippet': 'Figure 3: Most general definition of overfitting.  Now we can go back to cross-validation.  While ex'}, {'page': 28, 'score': 0.1898, 'snippet': 'Cross-validation is the first and most essential step when it comes to building machine learning mod'}]}


In [ ]:
# Test 1 — empty question
response = requests.post(
    "http://localhost:8000/query",
    json={"question": ""}
)
print("Empty:", response.status_code, response.json())

# Test 2 — out of domain question
response = requests.post(
    "http://localhost:8000/query",
    json={"question": "what is the capital of France"}
)
print("OOD:", response.status_code, response.json())

# Test 3 — valid question
response = requests.post(
    "http://localhost:8000/query",
    json={"question": "what is overfitting"}
)
print("Valid:", response.status_code, response.json())

Empty: 422 {'detail': 'Question cannot be empty'}
OOD: 200 {'answer': 'NO relevant content found for your question', 'topic': 'unknown', 'is_in_document': False, 'sources': []}
Valid: 200 {'answer': 'Overfitting occurs when a model fits perfectly on the training set but performs poorly on the test set, indicating that it learns the training data well but does not generalize to unseen samples. Another aspect of overfitting is when the test loss increases as the training loss improves.', 'topic': 'overfitting', 'is_in_document': True, 'sources': [{'page': 19, 'score': 0.1552, 'snippet': 'Approaching (Almost) Any Machine Learning Problem – Abhishek Thakur \n 20 \nThis is called overfitting'}, {'page': 20, 'score': 0.175, 'snippet': 'Figure 3: Most general definition of overfitting.  Now we can go back to cross-validation.  While ex'}, {'page': 19, 'score': 0.1765, 'snippet': 'Figure 2: Training and test accuracies for different values of max_depth.  One might argue that this'}, {'page': 1